[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/computation.ipynb)

# Computational Numerical Methods

This notebook implements fundamental numerical algorithms **from scratch** and
verifies their theoretical convergence properties.

**Contents:**
1. Floating-point errors -- catastrophic cancellation, machine epsilon
2. Root finding -- Bisection, Newton-Raphson, Secant (from scratch)
3. Convergence rate comparison of all three root-finders
4. Interpolation -- Lagrange, Runge's phenomenon, cubic splines
5. Numerical integration -- Trapezoidal rule, Simpson's rule
6. Numerical differentiation -- forward, central, backward finite differences

All algorithms are implemented from first principles using only NumPy for array
operations and Matplotlib for visualization. No SciPy solvers are called.

**Reference:** See `theory.md` in this folder for the mathematical foundations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

## 1. Floating-Point Errors

### Machine Epsilon

Machine epsilon $\varepsilon_{\text{mach}}$ is the smallest number such that
$\operatorname{fl}(1 + \varepsilon_{\text{mach}}) > 1$ in floating-point arithmetic.

For double precision (float64): $\varepsilon_{\text{mach}} = 2^{-52} \approx 2.22 \times 10^{-16}$.

### Catastrophic Cancellation

When two nearly equal numbers are subtracted, most significant digits cancel,
leaving only rounding errors. Example: computing $(1 - \cos x)/x^2$ for small $x$.

In [ ]:
# -- Machine epsilon computation --
# Algorithm: repeatedly halve eps until 1.0 + eps == 1.0
eps = 1.0
while (1.0 + eps) > 1.0:
    eps /= 2.0
# The last eps that DID make a difference is eps * 2
machine_eps_computed = eps * 2.0

print('Computed machine epsilon:  ', machine_eps_computed)
print('numpy machine epsilon:     ', np.finfo(np.float64).eps)
print('Theoretical 2^(-52):       ', 2**(-52))
print()

# -- Catastrophic cancellation demonstration --
# f(x) = (1 - cos(x)) / x^2  -->  true limit as x->0 is 1/2
# Direct evaluation loses precision for small x due to 1 - cos(x) ~ 0
x_vals = np.logspace(-1, -16, 16)
direct = (1.0 - np.cos(x_vals)) / x_vals**2
# Stable alternative via trig identity: 1 - cos(x) = 2*sin^2(x/2)
stable = 2.0 * (np.sin(x_vals / 2.0) / x_vals)**2

print(f"{'x':>20s} {'Direct':>22s} {'Stable':>22s} {'True':>10s}")
print('-' * 78)
for x, d, s in zip(x_vals, direct, stable):
    print(f'{x:20.2e} {d:22.16e} {s:22.16e} {0.5:10.4f}')

In [ ]:
# -- Visualization of catastrophic cancellation --
fig, ax = plt.subplots()
ax.semilogx(x_vals, direct, 'ro-', label='Direct: (1 - cos x) / x\u00b2')
ax.semilogx(x_vals, stable, 'bs-', label='Stable: 2 sin\u00b2(x/2) / x\u00b2')
ax.axhline(0.5, color='green', linestyle='--', linewidth=1.5, label='True value = 0.5')
ax.set_xlabel('x')
ax.set_ylabel('Computed value')
ax.set_title('Catastrophic Cancellation Demonstration')
ax.legend()
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()

## 2. Root Finding

We solve $f(x) = x^3 - 2x - 5 = 0$ using three methods implemented from scratch.
This equation has a real root near $x \approx 2.0946$.

### 2.1 Bisection Method

**Algorithm:** Given $f(a)f(b) < 0$, repeatedly halve the bracket $[a, b]$.

**Convergence:** Linear -- error bound halves each iteration:
$$|x_n - x^*| \le \frac{b - a}{2^n}$$

In [ ]:
def bisection(f, a, b, tol=1e-14, max_iter=100):
    # Bisection method with iteration tracking.
    # f: target function
    # a, b: bracket with f(a)*f(b) < 0
    # Returns (root, history)
    if f(a) * f(b) >= 0:
        raise ValueError('f(a) and f(b) must have opposite signs')
    history = []
    for i in range(max_iter):
        c = (a + b) / 2.0
        fc = f(c)
        history.append({'iter': i, 'a': a, 'b': b, 'c': c,
                        'f(c)': fc, 'interval': b - a})
        if abs(fc) < 1e-16 or (b - a) / 2.0 < tol:
            break
        if f(a) * fc < 0:
            b = c
        else:
            a = c
    return c, history


# Test function: f(x) = x^3 - 2x - 5
f = lambda x: x**3 - 2*x - 5
df = lambda x: 3*x**2 - 2  # derivative for Newton's method

root_bi, hist_bi = bisection(f, 2.0, 3.0)
print(f'Bisection root: {root_bi:.15f}')
print(f'Iterations:     {len(hist_bi)}')
print(f'f(root):        {f(root_bi):.2e}')

### 2.2 Newton-Raphson Method

**Update rule:** $x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$

**Convergence:** Quadratic near a simple root -- the number of correct digits
roughly **doubles** each iteration:
$$|x_{n+1} - x^*| \le C|x_n - x^*|^2$$

In [ ]:
def newton_raphson(f, df, x0, tol=1e-14, max_iter=100):
    # Newton-Raphson method with iteration tracking.
    # f: target function, df: derivative of f
    # x0: initial guess
    # Returns (root, history)
    history = []
    x = x0
    for i in range(max_iter):
        fx = f(x)
        dfx = df(x)
        if abs(dfx) < 1e-16:
            raise ValueError(f'Derivative near zero at iteration {i}')
        x_new = x - fx / dfx
        history.append({'iter': i, 'x': x, 'f(x)': fx, 'x_new': x_new})
        if abs(x_new - x) < tol:
            x = x_new
            break
        x = x_new
    return x, history


root_nw, hist_nw = newton_raphson(f, df, x0=2.5)
print(f'Newton root:    {root_nw:.15f}')
print(f'Iterations:     {len(hist_nw)}')
print(f'f(root):        {f(root_nw):.2e}')

# Verify quadratic convergence: ratio |e_{n+1}| / |e_n|^2 should approach constant
true_root = root_nw  # use Newton's high-accuracy result as reference
errors_nw = [abs(h['x'] - true_root) for h in hist_nw]
print('\nQuadratic convergence verification:')
print(f"{'Iter':>4s} {'|error|':>14s} {'|e_{n+1}|/|e_n|^2':>20s}")
for i in range(len(errors_nw) - 1):
    if errors_nw[i] > 1e-15:
        ratio = errors_nw[i + 1] / errors_nw[i]**2 if errors_nw[i] > 0 else float('inf')
        print(f'{i:4d} {errors_nw[i]:14.6e} {ratio:20.6f}')

### 2.3 Secant Method

**Update rule:** Replace $f'(x_n)$ with a finite-difference approximation:
$$x_{n+1} = x_n - f(x_n) \frac{x_n - x_{n-1}}{f(x_n) - f(x_{n-1})}$$

**Convergence:** Superlinear with order $\varphi = \frac{1+\sqrt{5}}{2} \approx 1.618$.

In [ ]:
def secant_method(f, x0, x1, tol=1e-14, max_iter=100):
    # Secant method with iteration tracking.
    # f: target function
    # x0, x1: two initial guesses
    # Returns (root, history)
    history = []
    for i in range(max_iter):
        f0, f1 = f(x0), f(x1)
        if abs(f1 - f0) < 1e-16:
            break
        x_new = x1 - f1 * (x1 - x0) / (f1 - f0)
        history.append({'iter': i, 'x0': x0, 'x1': x1, 'x_new': x_new, 'f(x1)': f1})
        if abs(x_new - x1) < tol:
            x1 = x_new
            break
        x0, x1 = x1, x_new
    return x1, history


root_sc, hist_sc = secant_method(f, 2.0, 3.0)
print(f'Secant root:    {root_sc:.15f}')
print(f'Iterations:     {len(hist_sc)}')
print(f'f(root):        {f(root_sc):.2e}')

### 2.4 Convergence Rate Comparison

Plotting $|x_n - x^*|$ vs iteration on a log scale for all three methods.
- **Bisection:** linear convergence (straight line on semilog plot)
- **Newton:** quadratic convergence (error drops dramatically)
- **Secant:** superlinear convergence (between linear and quadratic)

In [ ]:
# Compute errors for each method using Newton's result as true root
true_root = root_nw

errors_bisect = [abs(h['c'] - true_root) for h in hist_bi]
errors_newton = [abs(h['x'] - true_root) for h in hist_nw]
errors_secant = [abs(h['x1'] - true_root) for h in hist_sc]

# Remove trailing zeros (machine precision floor)
def trim_errors(errs, floor=1e-16):
    return [e for e in errs if e > floor]

eb = trim_errors(errors_bisect)
en = trim_errors(errors_newton)
es = trim_errors(errors_secant)

fig, ax = plt.subplots()
ax.semilogy(range(len(eb)), eb, 'ro-', label=f'Bisection ({len(hist_bi)} iters)', markersize=5)
ax.semilogy(range(len(en)), en, 'bs-', label=f'Newton ({len(hist_nw)} iters)', markersize=5)
ax.semilogy(range(len(es)), es, 'g^-', label=f'Secant ({len(hist_sc)} iters)', markersize=5)
ax.set_xlabel('Iteration')
ax.set_ylabel('|error|')
ax.set_title('Convergence Rate Comparison: Root Finding Methods\n$f(x) = x^3 - 2x - 5$')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nAll methods converge to root x* = {true_root:.15f}')
print(f'  Bisection: {len(hist_bi)} iterations (linear convergence)')
print(f'  Newton:    {len(hist_nw)} iterations (quadratic convergence)')
print(f'  Secant:    {len(hist_sc)} iterations (superlinear convergence, order ~ 1.618)')

## 3. Interpolation

### 3.1 Lagrange Interpolation

The unique polynomial of degree $\le n$ through $n+1$ points is:
$$p(x) = \sum_{i=0}^{n} y_i L_i(x), \qquad L_i(x) = \prod_{\substack{j=0 \\ j \ne i}}^{n} \frac{x - x_j}{x_i - x_j}$$

**Key property:** $L_i(x_j) = \delta_{ij}$ -- each basis polynomial is 1 at its own node and 0 at all others.

In [ ]:
def lagrange_basis(x, nodes, i):
    # Evaluate the i-th Lagrange basis polynomial L_i(x).
    # x: evaluation points, nodes: interpolation nodes, i: basis index
    x = np.asarray(x, dtype=float)
    n = len(nodes)
    L = np.ones_like(x)
    for j in range(n):
        if j != i:
            L *= (x - nodes[j]) / (nodes[i] - nodes[j])
    return L


def lagrange_interpolation(x, nodes, values):
    # Evaluate Lagrange interpolating polynomial at points x.
    # nodes: interpolation nodes x_i, values: function values y_i = f(x_i)
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    for i in range(len(nodes)):
        result += values[i] * lagrange_basis(x, nodes, i)
    return result


# Demo: interpolate sin(x) with 5 equally spaced nodes on [0, pi]
nodes = np.linspace(0, np.pi, 5)
values = np.sin(nodes)
x_fine = np.linspace(0, np.pi, 200)

p_interp = lagrange_interpolation(x_fine, nodes, values)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: interpolation result
ax1.plot(x_fine, np.sin(x_fine), 'k-', linewidth=2, label='sin(x)')
ax1.plot(x_fine, p_interp, 'r--', linewidth=2, label='Lagrange P4(x)')
ax1.plot(nodes, values, 'ko', markersize=8, zorder=5, label='Nodes')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Lagrange Interpolation of sin(x)')
ax1.legend()

# Right: basis polynomials
colors = plt.cm.Set1(np.linspace(0, 1, len(nodes)))
for i in range(len(nodes)):
    Li = lagrange_basis(x_fine, nodes, i)
    ax2.plot(x_fine, Li, color=colors[i], label=f'$L_{i}(x)$')
    ax2.plot(nodes[i], 1.0, 'o', color=colors[i], markersize=8, zorder=5)
ax2.axhline(0, color='gray', linewidth=0.5)
ax2.set_xlabel('x')
ax2.set_ylabel('$L_i(x)$')
ax2.set_title('Lagrange Basis Polynomials')
ax2.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 3.2 Runge's Phenomenon

High-degree polynomial interpolation on **equally spaced** nodes produces wild
oscillations near the boundaries of the interval. The classic example is:
$$f(x) = \frac{1}{1 + 25x^2}$$

**Chebyshev nodes** cluster near the endpoints and eliminate this issue:
$$x_k = \cos\left(\frac{2k + 1}{2(n+1)}\pi\right), \qquad k = 0, 1, \ldots, n$$

In [ ]:
def runge_function(x):
    # Runge's function: 1 / (1 + 25x^2)
    return 1.0 / (1.0 + 25.0 * x**2)


def chebyshev_nodes(n, a=-1, b=1):
    # Generate n+1 Chebyshev nodes on [a, b]
    k = np.arange(n + 1)
    nodes = np.cos((2*k + 1) / (2*(n + 1)) * np.pi)
    # Map from [-1, 1] to [a, b]
    return 0.5 * (a + b) + 0.5 * (b - a) * nodes


x_fine = np.linspace(-1, 1, 500)
f_true = runge_function(x_fine)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for deg in [5, 10, 15]:
    # Equally spaced nodes
    eq_nodes = np.linspace(-1, 1, deg + 1)
    eq_vals = runge_function(eq_nodes)
    p_eq = lagrange_interpolation(x_fine, eq_nodes, eq_vals)
    axes[0].plot(x_fine, p_eq, label=f'n = {deg}')

    # Chebyshev nodes
    ch_nodes = chebyshev_nodes(deg)
    ch_vals = runge_function(ch_nodes)
    p_ch = lagrange_interpolation(x_fine, ch_nodes, ch_vals)
    axes[1].plot(x_fine, p_ch, label=f'n = {deg}')

for ax in axes:
    ax.plot(x_fine, f_true, 'k-', linewidth=2.5, label='$f(x) = 1/(1+25x^2)$')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(fontsize=9)
    ax.set_ylim(-1.5, 2.0)

axes[0].set_title("Equally Spaced Nodes (Runge's Phenomenon)")
axes[1].set_title('Chebyshev Nodes (No Oscillations)')
plt.tight_layout()
plt.show()

### 3.3 Cubic Spline Interpolation (from scratch)

A **natural cubic spline** $S(x)$ is a piecewise cubic polynomial satisfying:
1. $S(x_i) = y_i$ (interpolation)
2. $S, S', S''$ are continuous across all interior nodes
3. $S''(x_0) = S''(x_n) = 0$ (natural boundary conditions)

This leads to a **tridiagonal linear system** for the second derivatives $M_i = S''(x_i)$,
which we solve using the Thomas algorithm (tridiagonal solver).

In [ ]:
def cubic_spline_natural(x_nodes, y_nodes):
    # Compute natural cubic spline coefficients.
    # Solves tridiagonal system for M_i = S''(x_i) using Thomas algorithm.
    # Returns list of (a, b, c, d) for each interval:
    #   S_i(x) = a + b*(x-x_i) + c*(x-x_i)^2 + d*(x-x_i)^3
    n = len(x_nodes) - 1
    h = np.diff(x_nodes)  # h[i] = x[i+1] - x[i]

    if n < 2:
        M = np.zeros(n + 1)
    else:
        # Right-hand side
        rhs = np.zeros(n - 1)
        for i in range(1, n):
            rhs[i - 1] = 6.0 * ((y_nodes[i+1] - y_nodes[i]) / h[i]
                                 - (y_nodes[i] - y_nodes[i-1]) / h[i-1])

        # Tridiagonal matrix coefficients
        lower = h[:-1].copy()
        diag = 2.0 * (h[:-1] + h[1:])
        upper = h[1:].copy()

        # Thomas algorithm -- forward elimination
        nn = len(diag)
        for i in range(1, nn):
            factor = lower[i - 1] / diag[i - 1]
            diag[i] -= factor * upper[i - 1]
            rhs[i] -= factor * rhs[i - 1]

        # Back substitution
        M_interior = np.zeros(nn)
        M_interior[-1] = rhs[-1] / diag[-1]
        for i in range(nn - 2, -1, -1):
            M_interior[i] = (rhs[i] - upper[i] * M_interior[i+1]) / diag[i]

        M = np.zeros(n + 1)
        M[1:-1] = M_interior

    # Compute coefficients for each piece
    coeffs = []
    for i in range(n):
        a_i = y_nodes[i]
        c_i = M[i] / 2.0
        d_i = (M[i+1] - M[i]) / (6.0 * h[i])
        b_i = ((y_nodes[i+1] - y_nodes[i]) / h[i]
               - h[i] * (2.0*M[i] + M[i+1]) / 6.0)
        coeffs.append((a_i, b_i, c_i, d_i))
    return coeffs


def eval_cubic_spline(x_eval, x_nodes, coeffs):
    # Evaluate cubic spline at given points.
    x_eval = np.asarray(x_eval, dtype=float)
    y_eval = np.zeros_like(x_eval)
    for k, xv in enumerate(x_eval):
        i = np.searchsorted(x_nodes, xv, side='right') - 1
        i = max(0, min(i, len(coeffs) - 1))
        a, b, c, d = coeffs[i]
        dx = xv - x_nodes[i]
        y_eval[k] = a + b*dx + c*dx**2 + d*dx**3
    return y_eval


# Compare Lagrange vs Cubic Spline on Runge's function (11 equally spaced nodes)
n_nodes = 11
x_nodes = np.linspace(-1, 1, n_nodes)
y_nodes = runge_function(x_nodes)
x_fine = np.linspace(-1, 1, 500)

p_lagrange = lagrange_interpolation(x_fine, x_nodes, y_nodes)

spline_coeffs = cubic_spline_natural(x_nodes, y_nodes)
p_spline = eval_cubic_spline(x_fine, x_nodes, spline_coeffs)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_fine, runge_function(x_fine), 'k-', linewidth=2.5, label='$f(x) = 1/(1+25x^2)$')
ax.plot(x_fine, p_lagrange, 'r--', linewidth=1.5, label=f'Lagrange (degree {n_nodes - 1})')
ax.plot(x_fine, p_spline, 'b-', linewidth=2, label='Natural cubic spline')
ax.plot(x_nodes, y_nodes, 'ko', markersize=7, zorder=5, label='Data points')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title("Lagrange vs Cubic Spline on Runge's Function (11 nodes)")
ax.legend()
ax.set_ylim(-1.0, 1.5)
plt.tight_layout()
plt.show()

print('Max |error| Lagrange:', np.max(np.abs(p_lagrange - runge_function(x_fine))))
print('Max |error| Spline:  ', np.max(np.abs(p_spline - runge_function(x_fine))))

## 4. Numerical Integration

### 4.1 Trapezoidal Rule (from scratch)

Approximate $\int_a^b f(x)\,dx$ using piecewise linear interpolation:

$$T_n = \frac{h}{2}\left[f(x_0) + 2f(x_1) + \cdots + 2f(x_{n-1}) + f(x_n)\right], \qquad h = \frac{b-a}{n}$$

**Error:** $O(h^2)$, i.e., doubling $n$ reduces error by factor $\approx 4$.

In [ ]:
def trapezoidal_rule(f, a, b, n):
    # Composite trapezoidal rule from scratch.
    # f: integrand, a,b: limits, n: number of subintervals
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    return h * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])


# Test: integrate e^(-x^2) from 0 to 1
integrand = lambda x: np.exp(-x**2)

# Reference value (known to high precision)
I_ref = 0.7468241328124271

n_values = [4, 8, 16, 32, 64, 128, 256, 512, 1024]
trap_errors = []

print(f"{'n':>6s} {'T_n':>18s} {'|Error|':>14s}")
print('-' * 42)
for n in n_values:
    T = trapezoidal_rule(integrand, 0, 1, n)
    err = abs(T - I_ref)
    trap_errors.append(err)
    print(f'{n:6d} {T:18.14f} {err:14.2e}')

# Error convergence plot
fig, ax = plt.subplots()
ax.loglog(n_values, trap_errors, 'ro-', label='Trapezoidal error')
n_arr = np.array(n_values, dtype=float)
ax.loglog(n_arr, trap_errors[0] * (n_values[0] / n_arr)**2, 'k--', alpha=0.5,
          label='$O(1/n^2)$ reference')
ax.set_xlabel('Number of subintervals $n$')
ax.set_ylabel('|Error|')
ax.set_title('Trapezoidal Rule: Error Convergence\n$\\int_0^1 e^{-x^2}dx$')
ax.legend()
plt.tight_layout()
plt.show()

### 4.2 Simpson's Rule (from scratch)

Uses piecewise quadratic interpolation ($n$ must be even):

$$S_n = \frac{h}{3}\left[f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + \cdots + 4f(x_{n-1}) + f(x_n)\right]$$

**Error:** $O(h^4)$ -- **two orders** better than trapezoidal!
Doubling $n$ reduces the error by factor $\approx 16$.

In [ ]:
def simpsons_rule(f, a, b, n):
    # Composite Simpson's rule from scratch.
    # n must be even.
    if n % 2 != 0:
        raise ValueError('n must be even for Simpson rule')
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    # Weights: 1, 4, 2, 4, 2, ..., 4, 1
    result = y[0] + y[-1]
    result += 4.0 * np.sum(y[1::2])    # odd indices
    result += 2.0 * np.sum(y[2:-1:2])  # even interior indices
    return h / 3.0 * result


# Compare trapezoidal vs Simpson error convergence
n_values_even = [4, 8, 16, 32, 64, 128, 256, 512, 1024]
simp_errors = []

print(f"{'n':>6s} {'Simpson':>18s} {'|Error|':>14s}")
print('-' * 42)
for n in n_values_even:
    S = simpsons_rule(integrand, 0, 1, n)
    err = abs(S - I_ref)
    simp_errors.append(err)
    print(f'{n:6d} {S:18.14f} {err:14.2e}')

# Side-by-side error comparison
fig, ax = plt.subplots()
n_arr = np.array(n_values_even, dtype=float)
ax.loglog(n_arr, trap_errors, 'ro-', label='Trapezoidal $O(h^2)$')
ax.loglog(n_arr, simp_errors, 'bs-', label="Simpson's $O(h^4)$")
ax.loglog(n_arr, trap_errors[0] * (n_values_even[0]/n_arr)**2, 'r--', alpha=0.3,
          label='$O(1/n^2)$ ref')
ax.loglog(n_arr, simp_errors[0] * (n_values_even[0]/n_arr)**4, 'b--', alpha=0.3,
          label='$O(1/n^4)$ ref')
ax.set_xlabel('Number of subintervals $n$')
ax.set_ylabel('|Error|')
ax.set_title("Trapezoidal vs Simpson's Rule: Error Convergence\n$\\int_0^1 e^{-x^2}dx$")
ax.legend()
plt.tight_layout()
plt.show()

idx64 = n_values_even.index(64)
print(f'\nAt n=64:')
print(f'  Trapezoidal error: {trap_errors[idx64]:.2e}')
print(f'  Simpson error:     {simp_errors[idx64]:.2e}')
print(f'  Simpson is {trap_errors[idx64]/simp_errors[idx64]:.0f}x more accurate')

## 5. Numerical Differentiation

### Finite Difference Formulas

Three standard approximations to $f'(x)$ using step size $h$:

| Formula | Expression | Error Order |
|---------|------------|-------------|
| Forward  | $\frac{f(x+h) - f(x)}{h}$ | $O(h)$ |
| Backward | $\frac{f(x) - f(x-h)}{h}$ | $O(h)$ |
| Central  | $\frac{f(x+h) - f(x-h)}{2h}$ | $O(h^2)$ |

For very small $h$, **roundoff error** dominates because we divide a tiny
numerator by a tiny denominator. The optimal $h$ balances truncation and roundoff.

In [ ]:
# Numerical differentiation: f(x) = sin(x) at x0 = pi/4
# Exact: f'(pi/4) = cos(pi/4) = sqrt(2)/2
x0 = np.pi / 4
exact_deriv = np.cos(x0)

h_values = np.logspace(-1, -15, 50)
forward_err = []
backward_err = []
central_err = []

for h in h_values:
    fwd = (np.sin(x0 + h) - np.sin(x0)) / h
    bwd = (np.sin(x0) - np.sin(x0 - h)) / h
    ctr = (np.sin(x0 + h) - np.sin(x0 - h)) / (2 * h)
    forward_err.append(abs(fwd - exact_deriv))
    backward_err.append(abs(bwd - exact_deriv))
    central_err.append(abs(ctr - exact_deriv))

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(h_values, forward_err, 'r-', label='Forward difference $O(h)$')
ax.loglog(h_values, backward_err, 'g--', label='Backward difference $O(h)$')
ax.loglog(h_values, central_err, 'b-', linewidth=2, label='Central difference $O(h^2)$')

# Reference slopes
ax.loglog(h_values, h_values * 0.5, 'k:', alpha=0.4, label='$O(h)$ reference')
ax.loglog(h_values, h_values**2 * 0.5, 'k-.', alpha=0.4, label='$O(h^2)$ reference')

ax.set_xlabel('Step size $h$')
ax.set_ylabel('|Error|')
ax.set_title('Numerical Differentiation of $\\sin(x)$ at $x = \\pi/4$\nError vs Step Size')
ax.legend(fontsize=10)
ax.set_xlim(1e-15, 1e-1)
plt.tight_layout()
plt.show()

# Table at selected h values
print(f"{'h':>12s} {'Forward err':>14s} {'Backward err':>14s} {'Central err':>14s}")
print('-' * 58)
for h in [1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12]:
    fwd = abs((np.sin(x0 + h) - np.sin(x0)) / h - exact_deriv)
    bwd = abs((np.sin(x0) - np.sin(x0 - h)) / h - exact_deriv)
    ctr = abs((np.sin(x0 + h) - np.sin(x0 - h)) / (2*h) - exact_deriv)
    print(f'{h:12.0e} {fwd:14.2e} {bwd:14.2e} {ctr:14.2e}')

print(f'\nNote: For very small h (< 1e-8), roundoff error dominates.')
print(f'Optimal h for forward/backward: ~sqrt(eps) = {np.sqrt(np.finfo(float).eps):.2e}')
print(f'Optimal h for central:          ~eps^(1/3) = {np.finfo(float).eps**(1/3):.2e}')

## Summary of Key Results

| Topic | Method | Key Property |
|-------|--------|-------------|
| Floating-point | Machine epsilon | $\varepsilon_{\text{mach}} = 2^{-52} \approx 2.22 \times 10^{-16}$ |
| Root finding | Bisection | Linear convergence, guaranteed |
| Root finding | Newton-Raphson | Quadratic convergence, requires $f'$ |
| Root finding | Secant | Superlinear (order $\varphi \approx 1.618$), no derivative needed |
| Interpolation | Lagrange | Unique polynomial through $n+1$ points |
| Interpolation | Cubic spline | Piecewise cubic, avoids Runge's phenomenon |
| Integration | Trapezoidal | Error $O(h^2)$ |
| Integration | Simpson's | Error $O(h^4)$ |
| Differentiation | Forward/backward | Error $O(h)$, optimal $h \sim \sqrt{\varepsilon}$ |
| Differentiation | Central | Error $O(h^2)$, optimal $h \sim \varepsilon^{1/3}$ |

**All algorithms were implemented from scratch** using only NumPy for array
operations. No SciPy solvers were called.

**Reference:** See `theory.md` for the full mathematical derivations.